# Exporting and Using Data as CSV

While Parquet is highly optimized for performance and type safety, many researchers and existing pipelines prefer or require CSV files. This notebook demonstrates how to easily convert the dataset into CSV format.

In [ ]:
import os
import sys
from pathlib import Path
import pandas as pd
import polars as pl

sys.path.append('..')
from loader import load_panel, load_metadata

# Ensure data directory exists
data_dir = Path("../data")
if not data_dir.exists():
    print("Data directory not found. Please run `python process_data.py` first.")

## 1. Exporting Metadata to CSV

In [ ]:
# Load Metadata Parquet using pandas directly
metadata_parquet_path = data_dir / "symbol_information.parquet"
metadata_csv_path = data_dir / "symbol_information.csv"

if metadata_parquet_path.exists():
    # Pandas reads parquet and exports to CSV natively
    df_meta = pd.read_parquet(metadata_parquet_path)
    df_meta.to_csv(metadata_csv_path, index=False)
    print(f"Successfully saved metadata CSV to {metadata_csv_path}")
else:
    print("Metadata parquet not found.")

## 2. Exporting Panel Data to CSV
If your dataset is large (e.g., 2GB+), converting the *entire* Parquet file to CSV will result in a massive file (10GB+). We recommend filtering down the universe or date range first using the provided Polars loader, then converting.

In [ ]:
try:
    # Use the lazy loader to grab a specific slice (e.g., BTC and ETH for 2023)
    lf = load_panel(start_date="2023-01-01", end_date="2023-03-01", symbols=["BTCUSDT", "ETHUSDT"])
    
    # Collect the slice into memory (Polars DataFrame)
    df_panel = lf.collect()
    
    # Write the Polars DataFrame directly to CSV
    panel_csv_path = data_dir / "panel_output_slice.csv"
    df_panel.write_csv(panel_csv_path)
    print(f"Successfully saved Panel slice CSV to {panel_csv_path}")
    
except Exception as e:
    print(f"Failed to load panel: {e}")

## 3. Alternative: Full Export using Pandas
If you want to dump the absolute raw `.parquet` file to CSV directly using Pandas, you can use the script below.

In [ ]:
# Find the latest parquet file
panel_files = sorted(data_dir.glob("binance_perps_panel_*.parquet"))

if panel_files:
    latest_panel = panel_files[-1]
    output_csv = data_dir / "full_panel_output.csv"
    
    # WARNING: This might use a lot of RAM depending on the dataset size
    print(f"Loading {latest_panel.name} via Pandas...")
    df_full = pd.read_parquet(latest_panel)
    print(f"Writing to {output_csv.name}...")
    df_full.to_csv(output_csv, index=False)
    print("Done!")
else:
    print("No panel parquet found.")